# HANDS-ON 01 · Gestión inicial de datos con Apache Spark
**UPN · Base de Datos Avanzadas y Big Data · Unidad III · Semana 11 · 4 horas**  
**Docente:** Dr. Eric Gustavo Coronel Castillo  
**Integrantes:** 1. ________ 2. ________ 3. ________ 4. ________  

**Contexto:** indicadores operativos simulados (agua, energía y residuos). Archivo de **100 000 filas**, siete columnas. **Ninguna observación representa datos reales**. Los registros fueron generados con semilla fija `20261103`.

**Producto:** notebook ejecutado, tabla de hallazgos, justificación de indicador y reflexión sustentada. En los RETOS se evalúan decisiones, no copiar instrucciones.

## 0. Antes de empezar
**Colab:** en la primera celda instala PySpark, luego sube `registros_operativos_simulados_100k.csv` usando Archivos. **Jupyter local:** omite instalación si ya tienes PySpark y coloca el CSV junto al cuaderno.  
Documentación: https://spark.apache.org/docs/latest/api/python/getting_started/install.html  
El entorno debe disponer de Java compatible con la versión de PySpark instalada. La sesión local de Spark no es un clúster multinodo.

In [ ]:
%pip -q install pyspark

In [ ]:
import sys, pyspark
print('Python:',sys.version.split()[0]); print('PySpark:',pyspark.__version__)

## 1. Situación problemática: plantear preguntas (20 min)
Una institución ficticia recibe 100 000 registros de metas y valores observados. La dirección requiere un indicador por sector, pero el archivo puede contener problemas de calidad.  
**Antes de ejecutar consultas:** plantea dos preguntas medibles, tres reglas de calidad y explica qué consecuencias tendría publicar un indicador incorrecto.

## 2. Crear SparkSession (ejemplo guiado; 35 min)
SparkSession es el punto de entrada a la API de DataFrames. `local[*]` emplea los recursos locales disponibles; **no prueba distribución entre varias máquinas**.

In [ ]:
from pyspark.sql import SparkSession
spark = (SparkSession.builder.master('local[*]')
         .appName('UPN_S11_Spark_100k').getOrCreate())
spark.sparkContext.setLogLevel('ERROR')
print('Spark:',spark.version,'| modo:',spark.sparkContext.master)

## 3. Carga y exploración (35 min)
Sube el CSV. La inferencia automática de tipos es conveniente para una introducción, pero no reemplaza un contrato de datos. Registra los tipos detectados y explica qué harías con un esquema fijo en producción.

In [ ]:
from pathlib import Path
rutas=[Path('registros_operativos_simulados_100k.csv'),Path('/content/registros_operativos_simulados_100k.csv')]
ruta=next((str(p) for p in rutas if p.is_file()),None)
assert ruta, 'Sube el archivo CSV a Colab o colócalo junto al cuaderno'
df=(spark.read.option('header',True).option('inferSchema',True).csv(ruta))
df.printSchema(); df.show(8,truncate=False)
print('Filas:',df.count(),'| columnas:',len(df.columns))
assert df.count()==100000 and len(df.columns)==7, 'Fuente equivocada o incompleta' 

## 4. Transformaciones y acciones (ejemplo guiado)
Identifica qué líneas preparan una transformación y cuáles provocan ejecutar el trabajo.

In [ ]:
df_norte=df.filter(df.zona=='Norte').select('id_registro','sector','meta_diaria')
print('Transformación definida (aún sin mostrar resultados)')
df_norte.show(5,truncate=False)

## 5. RETO A · Auditoría de calidad (55 min)
**Entregar** una tabla con hallazgo, cantidad de filas, cantidad de IDs afectados, ejemplo e impacto. Detecta: (1) observaciones nulas, (2) metas no positivas, (3) IDs repetidos, (4) sobrecumplimientos potencialmente válidos.  
**Pregunta de decisión:** ¿puede eliminarse automáticamente una fila cuya observación supera la meta?

In [ ]:
# RETO A.1 — Localizar filas con valor_observado ausente
# Tu código:

In [ ]:
# RETO A.2 — Metas no positivas y sobrecumplimientos
# Tu código:

In [ ]:
# RETO A.3 — Agrupar por ID para localizar duplicados
# Tu código:

## 6. RETO B · Análisis por sector (35 min)
Construye un conteo por sector y un cuadro de calidad por zona. Explica por qué los conteos previos a la depuración **no representan necesariamente operaciones únicas**. Considera al menos una alternativa de consulta.

In [ ]:
# RETO B — tabla por sector y cuadro de faltantes por zona
# Tu código:

## 7. RETO C · Corregir indicador engañoso (35 min)
**Propuesta a cuestionar:** `avg(valor_observado / meta_diaria)` sobre todo el CSV. ¿Por qué mezcla registros inválidos, duplica eventos y da el mismo peso a metas muy diferentes?  
Diseña una política explícita de calidad y compara al menos dos indicadores: (a) promedio simple de razones válidas y (b) razón entre sumas. Señala qué pregunta responde cada uno.

In [ ]:
# RETO C — escribe tu política de limpieza y dos indicadores por sector
# Tu código:

## 8. Validación, discusión y cierre (25 min)
Responde con evidencia: ¿qué diferencia hay entre una transformación (`filter`) y una acción (`count`, `show`)? ¿Por qué Spark local puede ser más lento que pandas para 100 000 filas? ¿Cuál es el impacto de duplicados sobre el indicador? ¿Cuál es la limitación de datos simulados? Declara cualquier uso de IA.

In [ ]:
# VALIDACIÓN DEL ESTUDIANTE — agrega al menos 3 asserts propios
assert df.count()==100000
assert len(df.columns)==7
# Tus verificaciones adicionales:

## Referencias
- Sílabo UPN 2026-2, Unidad III, semana 11 (Gestión de Datos con Spark).
- Documentación oficial: https://spark.apache.org/docs/latest/api/python/getting_started/install.html
- Guía Spark SQL y DataFrames: https://spark.apache.org/docs/latest/sql-getting-started.html

**Reto opcional:** ejecutar sobre un volumen mayor generado con el script complementario y comparar tiempos, documentando recursos del entorno. Evita conclusiones universales de rendimiento a partir de una sola ejecución.